# Testing the MC-PILCO Functionality

In [3]:
# %load ~/dev/marthaler/header.py
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
# Enable Float64 for more stable matrix inversions.
import jax
import equinox as eqx
from jax import Array, config
import jax.numpy as jnp
import numpy as np
import jax.random as jr
from jaxtyping import ArrayLike, install_import_hook, Array, Float, Int, PyTree  
from typing import Optional
import matplotlib as mpl
import matplotlib.pyplot as plt

config.update("jax_enable_x64", True)

cols = mpl.rcParams["axes.prop_cycle"].by_key()["color"]

In [5]:
from jax_mc_pilco.controllers import Controller, ZeroController
from jax_mc_pilco.rewards import unit_cost
from jax_mc_pilco.model_learning.dynamical_models import IMGPR
from jax_mc_pilco.policy_learning.rollout import policy_rollout

In [6]:
from IPython import display

## Globals

In [7]:
num_particles = 400
num_trials = 8
T_sampling = 0.05
T_exploration = 0.35
T_control = 3.0
sim_timestep = 0.01
starting_dropout_probability = 0.25
control_horizon = int(T_control / T_sampling)
num_basis = 200
umax = 2.0

In [9]:
key = jr.key(42)

In [35]:
state_dim = 3
action_dim = 2

In [36]:
key, subkey = jr.split(key)
states = 2 * jr.uniform(subkey, shape=(100,state_dim)) - 1
key, subkey = jr.split(key)
actions = 2 * jr.uniform(subkey, shape=(100,action_dim)) - 1

In [37]:
states.shape, actions.shape

((100, 3), (100, 2))

In [38]:
timesteps = np.linspace(0, 10,11)

In [41]:
policy = ZeroController(state_dim, action_dim, to_squash=True, max_action=1.0)

# Test Rollout

In [42]:
import gpjax

In [46]:
model = IMGPR(states=jnp.array(states),actions=jnp.array(actions),kernel_funcs=gpjax.kernels.RBF())

In [49]:
policy_rollout(
    policy,
    states[0:3,:],
    actions[0:3,:],
    model,
    timesteps,
    key,
    unit_cost)

IndexError: Too many indices: 1-dimensional array indexed with 2 regular indices.